# Pelatihan & Evaluasi Model Deteksi Stres Pasca-Aktivitas (StressGuard)

Notebook ini mendokumentasikan seluruh alur kerja *Machine Learning* untuk proyek **StressGuard Web App** (UAS Sistem Cerdas TI24317):
1. **Eksplorasi & Pembersihan Data** (`student_lifestyle_dataset.csv`).
2. **Seleksi Fitur** (Mempertahankan 5 fitur durasi harian; membuang `Student_ID` dan `GPA`).
3. **Train/Test Split** (80% data latih, 20% data uji terstratifikasi).
4. **Pelatihan Model** (`RandomForestClassifier` dengan `n_estimators=100`, `max_depth=6`, `random_state=42`).
5. **Evaluasi Model** (Akurasi Latih vs Uji, *Classification Report*, *Confusion Matrix*, dan *Feature Importance*).
6. **Ekspor Model** ke format `.joblib` untuk diintegrasikan ke Django.

In [ ]:
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# Kontrak Fitur: urutan 5 kolom ini wajib sama persis dengan backend Django (prediction/views.py)
FEATURE_COLUMNS = [
    "Study_Hours_Per_Day",
    "Extracurricular_Hours_Per_Day",
    "Sleep_Hours_Per_Day",
    "Social_Hours_Per_Day",
    "Physical_Activity_Hours_Per_Day",
]
TARGET_COLUMN = "Stress_Level"
CLASS_ORDER = ["Low", "Moderate", "High"]

## 1. Pemuatan & Eksplorasi Data (EDA)

In [ ]:
df = pd.read_csv("student_lifestyle_dataset.csv")
df.columns = df.columns.str.strip()

print("Ukuran Dataset:", df.shape)
print("Nilai Kosong:\n", df.isnull().sum())
print("\nJumlah Duplikat:", df.duplicated().sum())
print("\nDistribusi Kelas Target:\n", df[TARGET_COLUMN].value_counts())
df.head()

## 2. Seleksi Fitur & Pembagian Data (80:20 Stratified Split)

- `Student_ID` dibuang karena hanya berupa nomor urut identitas.
- `GPA` dibuang karena tujuan aplikasi adalah mengukur beban gaya hidup dan kelelahan harian pasca-aktivitas, bukan nilai akademik kumulatif.

In [ ]:
X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f"Data Latih : {len(X_train)} baris")
print(f"Data Uji   : {len(X_test)} baris")

## 3. Pelatihan Model Random Forest & Evaluasi Metrik

Parameter `max_depth=6` digunakan untuk membatasi kedalaman tiap pohon keputusan agar model tidak sekadar menghafal data latih (*overfitting*).

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=6,
    random_state=42,
)
model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

train_acc = accuracy_score(y_train, y_train_pred) * 100
test_acc = accuracy_score(y_test, y_test_pred) * 100

print(f"Akurasi Data Latih : {train_acc:.2f}%")
print(f"Akurasi Data Uji   : {test_acc:.2f}%")
print(f"Selisih (Gap)      : {abs(train_acc - test_acc):.2f}%\n")
print(classification_report(y_test, y_test_pred, labels=CLASS_ORDER, digits=4))

## 4. Visualisasi Confusion Matrix & Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=150)
fig.patch.set_facecolor("#F6F5F2")

cm = confusion_matrix(y_test, y_test_pred, labels=CLASS_ORDER)
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Greens",
    cbar=False,
    xticklabels=CLASS_ORDER,
    yticklabels=CLASS_ORDER,
    ax=axes[0],
)
axes[0].set_title("Confusion Matrix (Data Uji 20%)", fontweight="bold")
axes[0].set_xlabel("Prediksi Model")
axes[0].set_ylabel("Label Sebenarnya")

importance_df = pd.DataFrame({
    "Fitur": FEATURE_COLUMNS,
    "Bobot": model.feature_importances_,
}).sort_values("Bobot", ascending=True)

axes[1].barh(importance_df["Fitur"], importance_df["Bobot"], color="#1F2937")
axes[1].set_title("Feature Importance", fontweight="bold")
axes[1].set_xlabel("Skor Kepentingan Relatif")

plt.tight_layout()
plt.savefig("evaluasi_model.png", bbox_inches="tight")
plt.show()

## 5. Ekspor Model Terlatih ke File `.joblib`

In [ ]:
output_dir = Path("prediction/ml_models")
output_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(model, output_dir / "model_stres.joblib")
print("Model berhasil disimpan ke:", output_dir / "model_stres.joblib")